# Step 8: Memory & Sessions

        _Learner Notebook_  
        Estimated time: **60 minutes**  
        Difficulty: **Intermediate**

        ## Learning Objectives
        - [ ] Distinguish in-memory session state from external persistence.
- [ ] Store and retrieve conversation history in SQLite.
- [ ] Blend notebook-local session state with a durable store.
- [ ] Prepare for RAG and richer context engineering.

        ## Prerequisites
        - Step 7 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Sessions give you in-run context. Durable memory lets your system bring context back later. This notebook uses a lightweight SQLite store so the pattern stays easy to inspect locally.

Expected output and validation notes:

Expected output snapshot:

- Two or more conversation rows saved in SQLite
- A recall prompt that references Alice correctly


## Part 2: Core Implementation

### Persist simple conversation turns


In [ ]:
from agent_framework import AgentSession

memory = SQLiteConversationMemory(resolve_notebook_root() / "data" / "databases" / "memory_demo.sqlite3")
session = AgentSession(session_id="alice-session")
memory_agent = create_agent(
    name="MemoryAgent",
    instructions="You are a helpful assistant that uses prior context when available.",
)

first = await memory_agent.run("My name is Alice and I work on platform engineering.", session=session)
memory.save("alice", "My name is Alice and I work on platform engineering.", first.text)

second = await memory_agent.run("Give me one encouraging sentence about learning agents.", session=session)
memory.save("alice", "Give me one encouraging sentence about learning agents.", second.text)

history = memory.history("alice")
print_json([record.__dict__ for record in history], title="SQLite conversation history")


## Part 2: Core Implementation

### Blend memory back into a prompt


In [ ]:
recent_notes = "\n".join(f"- {item.user_message}" for item in history)
recall_prompt = f"""Use this known context about the user:
{recent_notes}

What do you know about Alice?"""

recall = await memory_agent.run(recall_prompt, session=session)
print(recall.text)


## Part 3: Hands-On Exercises

Store another turn and create a helper that formats the last three memory records for prompt injection.

Try the exercise yourself before looking at the solutions in Part 4.


In [ ]:
# TODO: save another interaction
# TODO: write a helper that returns the most recent three records as bullet points


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
third_question = "Please remember that I prefer concise examples."
third_response = await memory_agent.run(third_question, session=session)
memory.save("alice", third_question, third_response.text)

def recent_memory_bullets(user_id: str, limit: int = 3) -> str:
    rows = memory.history(user_id, limit=limit)
    return "\n".join(f"- {row.user_message}" for row in rows)

print(recent_memory_bullets("alice"))


## Part 5: Best Practices & Tips

        - Use session state for active conversations and external storage for recall later.
- Persist only the fields you truly need for later context.
- Review stored memory formats before you rely on them in prompts.


## Summary & Key Takeaways

You now have both short-lived session context and a simple durable memory layer.


## Additional Resources

        - `helpers/memory.py`
- `Step 3 notebook`
